# DenseNet — Densely Connected Convolutional Networks

**Citation:** Huang, G., Liu, Z., van der Maaten, L., & Weinberger, K. Q. (2017). Densely connected convolutional networks. *CVPR 2017*. arXiv:1608.06993.

**Registered names:** `densenet121`, `densenet169`, `densenet201`, `densenet264`

**Summary:** DenseNet connects each layer to ALL subsequent layers: the input to layer $ℓ$ is the concatenation of all previous outputs. This maximum feature reuse allows very narrow layers (growth rate k=32) to compete with much wider non-dense networks, winning Best Paper at CVPR 2017.

---

| Model | Blocks | Params | Top-1 |
|-------|--------|--------|-------|
| DenseNet-121 | (6,12,24,16) | 8.0M | 74.9% |
| DenseNet-169 | (6,12,32,32) | 14.1M | 76.2% |
| DenseNet-201 | (6,12,48,32) | 20.0M | 77.4% |
| DenseNet-264 | (6,12,64,48) | 33.4M | 77.9% |

## Architecture Diagram

```
Input [B, 3, 224, 224]
       │
 Conv(7×7,s2)+BN+ReLU+MaxPool  → [B, 64, H/4]
       │
╔══════════════════════════════════════════════╗
║  DenseBlock 1 (L=6 layers)                  ║
║                                              ║
║  x₀ ──────────────────────────────────────► ║
║  x₀ ──→ Layer₁ ──→ cat(x₀,x₁) ──► Layer₂  ║
║  cat(x₀,x₁,x₂) ──→ Layer₃ ──► ...          ║
║  Layer ℓ input = cat(x₀, x₁, ..., x_{ℓ-1})  ║
╚══════════════════════════════════════════════╝
       │
 Transition: BN→ReLU→Conv1×1(C→C//2)→AvgPool(2)
       │                        → H/8, channels ÷2
╔══════════════════════╗
║  DenseBlock 2 (L=12) ║  → H/16
╚══════════════════════╝
       │ Transition
╔══════════════════════╗
║  DenseBlock 3 (L=24) ║  → H/32
╚══════════════════════╝
       │ Transition
╔══════════════════════╗
║  DenseBlock 4 (L=16) ║
╚══════════════════════╝
       │
 BN → ReLU → GlobalAvgPool → pooled [B, 1024]

Each Dense Layer: BN→ReLU→Conv1×1(4k)→BN→ReLU→Conv3×3(k)  [k=32]
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.patch.set_facecolor('#0f1117')

# ─── Left: DenseBlock internal connectivity ───
ax = axes[0]
ax.set_facecolor('#0f1117')
ax.set_xlim(0, 8); ax.set_ylim(0, 10); ax.axis('off')
ax.text(4, 9.5, 'DenseBlock — All-Pairs Connectivity', ha='center', color='white', fontsize=12, fontweight='bold')

n_layers = 5
layer_colors = ['#4a9eff', '#6bcb77', '#ff9f43', '#ff6b9d', '#a29bfe']
ys = np.linspace(7.8, 1.5, n_layers)

for i, (y, color) in enumerate(zip(ys, layer_colors)):
    label = 'x₀ (input)' if i == 0 else f'Layer {i}\nBN→ReLU→Conv1×1→BN→ReLU→Conv3×3'
    ax.add_patch(FancyBboxPatch((1.0, y - 0.3), 6.0, 0.65,
                                 boxstyle='round,pad=0.07', facecolor=color,
                                 edgecolor='white', linewidth=0.7, alpha=0.85))
    ax.text(4.0, y + 0.02, label, ha='center', va='center', color='white', fontsize=8, fontweight='bold')

# Draw dense connections: every prior layer feeds every subsequent layer
for src in range(n_layers - 1):
    for dst in range(src + 1, n_layers):
        x_offset = 0.7 - dst * 0.12
        color = layer_colors[src]
        ax.annotate('', xy=(x_offset, ys[dst] + 0.3), xytext=(x_offset, ys[src] - 0.3),
                    arrowprops=dict(arrowstyle='->', color=color, lw=0.8, alpha=0.6))

ax.text(4, 0.5, 'Concat(x₀, x₁, ..., xₗ₋₁) fed as input to layer ℓ',
        ha='center', color='#cccccc', fontsize=8, style='italic')

# ─── Right: Full DenseNet flow ───
ax2 = axes[1]
ax2.set_facecolor('#0f1117')
ax2.set_xlim(0, 8); ax2.set_ylim(0, 10); ax2.axis('off')
ax2.text(4, 9.5, 'DenseNet-121 Full Architecture', ha='center', color='white', fontsize=12, fontweight='bold')

blocks = [
    ('Initial Conv\n7×7,s2+BN+ReLU+MaxPool\n64ch, H/4', '#555566', 8.5),
    ('DenseBlock 1\nL=6 layers, k=32\n→ 256ch, H/4', '#4a9eff', 7.0),
    ('Transition 1\nConv1×1 + AvgPool\n128ch, H/8', '#444455', 5.7),
    ('DenseBlock 2\nL=12 layers, k=32\n→ 512ch, H/8', '#6bcb77', 4.3),
    ('Transition 2\nConv1×1 + AvgPool\n256ch, H/16', '#444455', 3.0),
    ('DenseBlock 3\nL=24 layers, k=32\n→ 1024ch, H/16', '#ff9f43', 1.7),
    ('Transition 3 + DenseBlock4\n+ BN+ReLU+GlobalAvgPool\npooled [B, 1024]', '#a29bfe', 0.4),
]

for label, color, cy in blocks:
    ax2.add_patch(FancyBboxPatch((0.8, cy - 0.4), 6.4, 0.95,
                                  boxstyle='round,pad=0.08', facecolor=color,
                                  edgecolor='white', linewidth=0.8, alpha=0.88))
    ax2.text(4.0, cy + 0.08, label, ha='center', va='center', color='white', fontsize=7.5, fontweight='bold')

for i in range(len(blocks) - 1):
    y1 = blocks[i][2] - 0.4
    y2 = blocks[i+1][2] + 0.55
    ax2.annotate('', xy=(4.0, y2), xytext=(4.0, y1),
                arrowprops=dict(arrowstyle='->', color='white', lw=1.2))

plt.suptitle('DenseNet Architecture (Huang et al. 2017)', color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Key Innovations

### 1. Dense Connectivity
In a DenseBlock of $L$ layers, layer $ℓ$ receives input from all prior layers:
$$x_ℓ = H_ℓ([x_0, x_1, \ldots, x_{ℓ-1}])$$
where $[\cdot]$ denotes channel concatenation and $H_ℓ$ is BN→ReLU→Conv1×1→BN→ReLU→Conv3×3.

Total connections in a block: $\frac{L(L+1)}{2}$ vs $L$ in ResNet.

### 2. Growth Rate — Maximally Narrow Layers
Each layer adds only $k=32$ new channels (vs 64-256 in ResNet). This works because the growing concatenation already carries rich features. The total channels after $L$ layers:
$$C_L = C_0 + L \cdot k$$

### 3. Bottleneck for Efficiency
Each layer uses a 1×1 conv to compress the concatenated input to $4k$ channels before the 3×3 conv:
$$\text{DenseBottleneck: } \text{concat}(\cdot) \xrightarrow{\text{BN+ReLU+Conv1×1}} 4k \xrightarrow{\text{BN+ReLU+Conv3×3}} k$$

### 4. Compression via Transition Layers
Transition layers compress channels by $\theta=0.5$ and halve spatial resolution:
$$C_{out} = \lfloor \theta \cdot C_{in} \rfloor = C_{in} // 2$$

### 5. Feature Map Extraction for Dense Tasks
In this implementation, feature maps are captured BEFORE the Transition's AvgPool (at DenseBlock output resolution), providing multi-scale features compatible with FPN heads.

## Code Walkthrough

Implemented in `hale_core/nn/vision/cnn/densenet.py`.

**DenseBlock — concatenation accumulation:**
```python
class DenseBlock(nn.Module):
    def forward(self, x):
        features = [x]  # accumulate all outputs
        for layer in self.layers:
            new_feat = layer(torch.cat(features, dim=1))  # grows each call
            features.append(new_feat)
        return torch.cat(features, dim=1)  # full concatenated output
```

Memory note: the concatenation list grows to $L+1$ tensors per forward pass. For very deep blocks (L=64 in DenseNet-264), gradient checkpointing would reduce peak memory at the cost of recomputation.

**Transition — channel compression + spatial downsampling:**
```python
class _Transition(nn.Module):
    def __init__(self, in_ch, out_ch):  # out_ch = in_ch // 2
        self.norm = nn.BatchNorm2d(in_ch)
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        self.pool = nn.AvgPool2d(kernel_size=2, stride=2)
```

In [ ]:
import torch
import sys
sys.path.insert(0, '../../src')

from hale_core.nn.vision.registry import build_vision_backbone

for name in ['densenet121', 'densenet169', 'densenet201']:
    bb = build_vision_backbone(name, img_size=224)
    bb.eval()
    x = torch.randn(1, 3, 224, 224)
    with torch.no_grad():
        out = bb(x)
    fmaps = [tuple(f.shape) for f in out.feature_maps]
    print(f'{name}:')
    print(f'  pooled:       {tuple(out.pooled_output.shape)}')
    print(f'  feature_maps: {fmaps}')
    print(f'  params:       {bb.num_parameters()/1e6:.1f}M')
    print()

## References

| Paper | Venue | Key Contribution |
|-------|-------|------------------|
| Huang et al. (2017). Densely Connected CNNs. | CVPR 2017 (Best Paper) | DenseNet — all-pairs skip connections |
| He et al. (2016). Deep Residual Learning. | CVPR 2016 | ResNet — single skip connection predecessor |
| Srivastava et al. (2015). Highway Networks. | NeurIPS 2015 | Gated skip connections |
| Huang et al. (2016). Deep Networks with Stochastic Depth. | ECCV 2016 | Stochastic depth (same group) |
| Pleiss et al. (2017). Memory-Efficient Implementation of DenseNets. | arXiv 2017 | Gradient checkpointing for DenseBlocks |

**Full Citation:**
> Huang, G., Liu, Z., van der Maaten, L., & Weinberger, K. Q. (2017). Densely connected convolutional networks. In *Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition* (pp. 4700–4708). arXiv:1608.06993.